# LLM Grammar Experiment - Colab Runner
Make sure you're using a **GPU runtime**: Runtime > Change runtime type > T4 GPU

In [ ]:
# Step 1: Check GPU is available
!nvidia-smi

In [ ]:
# Step 2: Clone your repo (replace YOUR_GITHUB_URL with your actual repo URL)
!git clone https://github.com/molybrine/Bsc-project.git
%cd llm_gramer
!git checkout master

In [ ]:
# Step 3: Install dependencies
!pip install -q torch transformers accelerate bitsandbytes sentencepiece
!pip install -q numpy pandas python-Levenshtein scipy statsmodels pingouin matplotlib seaborn tqdm pyyaml

In [ ]:
# Step 4: Verify GPU and VRAM
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"VRAM: {total:.1f} GB")

In [ ]:
# Step 5: Patch max_memory for Colab T4 (16GB VRAM)
# The repo default is 8GiB for a 6GB laptop GPU — T4 can use much more
path = 'model_inference.py'
with open(path) as f:
    code = f.read()
code = code.replace(
    "max_memory={0: '8GiB', 'cpu': '16GiB'}",
    "max_memory={0: '14GiB', 'cpu': '16GiB'}"
)
with open(path, 'w') as f:
    f.write(code)
print("Patched max_memory: 8GiB -> 14GiB for T4 GPU")

In [ ]:
# Step 6: Run bloomz at 8-bit
!python run_experiment.py --model bloomz --quantize 8bit

In [ ]:
# Step 7: Run pythia at 8-bit
!python run_experiment.py --model pythia --quantize 8bit

In [ ]:
# Step 8: Merge results and run full analysis (stats + figures)
import glob
bloomz_csv = sorted(glob.glob('results/results_bloomz_*.csv'))[-1]
pythia_csv = sorted(glob.glob('results/results_pythia_*.csv'))[-1]
print(f"Merging:\n  {pythia_csv}\n  {bloomz_csv}")
!python merge_and_analyse.py --model1 "{pythia_csv}" --model2 "{bloomz_csv}"

In [ ]:
# Step 9: View results summary
import pandas as pd
import glob

for f in sorted(glob.glob('results/*.csv')):
    df = pd.read_csv(f)
    print(f"\n{'='*50}")
    print(f"File: {f}")
    print(f"Instances: {len(df)}")
    print(f"Overall accuracy: {df['exact_match'].mean():.3f}")
    print(f"Mean edit distance: {df['edit_distance'].mean():.3f}")
    print(f"\nAccuracy by variant:")
    print(df.groupby('variant')['exact_match'].mean())
    print(f"\nAccuracy by shot count:")
    print(df.groupby('shot_count')['exact_match'].mean())

In [ ]:
# Step 10: Display generated figures
from IPython.display import Image, display
import os

figures = ['learning_curves.png', 'interaction_plot.png', 'heatmap.png', 'error_analysis.png']
for fig in figures:
    path = f'figures/{fig}'
    if os.path.exists(path):
        print(f"\n{'='*50}\n{fig}\n{'='*50}")
        display(Image(filename=path))
    else:
        print(f"Missing: {path}")

In [ ]:
# Step 11: Download results and figures
from google.colab import files
import glob

for f in glob.glob('results/*.csv'):
    files.download(f)
for f in glob.glob('figures/*.png'):
    files.download(f)